# Part 2. 취소 예측

In [1]:
# 기본 구조 확인하기
import pandas as pd
import numpy as np

df = pd.read_csv('hotel_bookings.csv')

In [2]:
# 날짜 칼럼 재정의
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-'
    + df['arrival_date_month'] + '-'
    + df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02,2015-07-01
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03,2015-07-01


In [3]:
# ===== Part 2: 취소예측 모델링 =====

# 1) 예약일 생성 (투숙일 - 리드타임) — 시간순 분할용
df['booking_date'] = pd.to_datetime(df['arrival_date']) - pd.to_timedelta(df['lead_time'], unit='D')

print("예약일 범위:", df['booking_date'].min().date(), "~", df['booking_date'].max().date())
print("\n예약일 기준 연도별 건수:")
print(df['booking_date'].dt.year.value_counts().sort_index())

예약일 범위: 2013-06-24 ~ 2017-08-31

예약일 기준 연도별 건수:
booking_date
2013        1
2014     2636
2015    31645
2016    58543
2017    26565
Name: count, dtype: int64


In [4]:
# 2) 시간순 분할 (예약일 기준 70/15/15 분위수 컷)
df = df.sort_values('booking_date').reset_index(drop=True)

q70 = df['booking_date'].quantile(0.70)
q85 = df['booking_date'].quantile(0.85)
print("분할 경계:", q70.date(), "|", q85.date())

train = df[df['booking_date'] <= q70]
valid = df[(df['booking_date'] > q70) & (df['booking_date'] <= q85)]
test  = df[df['booking_date'] > q85]


# 분할별 타겟 분포 — 시간에 따른 취소율 드리프트 확인
for name, d in [('train', train), ('valid', valid), ('test', test)]:
    print(f"{name}: {len(d):>6} ({len(d)/len(df)*100:.1f}%)  "
          f"{d['booking_date'].min().date()} ~ {d['booking_date'].max().date()}  "
          f"취소율 {d['is_canceled'].mean():.3f}")

분할 경계: 2016-11-06 | 2017-02-03


train:  83580 (70.0%)  2013-06-24 ~ 2016-11-06  취소율 0.380
valid:  18041 (15.1%)  2016-11-07 ~ 2017-02-03  취소율 0.400
test:  17769 (14.9%)  2017-02-04 ~ 2017-08-31  취소율 0.296


In [5]:
# test 취소율이 낮은 이유 확인 — 리드타임 분포와 투숙 월 분포 비교
for name, d in [('train', train), ('test', test)]:
    print(f"{name}: 리드타임 중앙값 {d['lead_time'].median():.0f}일, "
          f"평균 {d['lead_time'].mean():.0f}일")

print("\ntest 투숙 월 분포:")
print(test['arrival_date'].dt.month.value_counts().sort_index())

train: 리드타임 중앙값 81일, 평균 119일
test: 리드타임 중앙값 29일, 평균 49일

test 투숙 월 분포:
arrival_date
2    1131
3    1945
4    2146
5    2977
6    2957
7    3044
8    3569
Name: count, dtype: int64


In [6]:
# ===== 3) 전처리 (함수로 통합) =====
# 원본 df는 그대로 두고, 분할된 train/valid/test에 동일 처리 적용

def preprocess(d):
    d = d.copy()
    
    # (1) 누수 컬럼 제거
    #  - reservation_status(_date): 취소 결과 그 자체 → 정답 누수
    #  - assigned_room_type: 체크인 시점 확정 → 예측 시점엔 모름
    #  - required_car_parking_spaces: 취소자 0명 → 이용시점 정보 누수
    #  - 날짜/분할용: 절대날짜 과적합 방지
    drop_cols = ['reservation_status', 'reservation_status_date',
                 'assigned_room_type', 'required_car_parking_spaces',
                 'arrival_date', 'booking_date', 'booking_ym', 'company',
                 # 예약 이후에 값이 쌓이는 최종값: 예약/투숙 7일 전 시점에는 알 수 없음 (시점 누수)
                 'booking_changes', 'total_of_special_requests', 'days_in_waiting_list',
                 # 시간 외삽 방지: 미래 연도는 학습에 없는 값
                 'arrival_date_year']
    d = d.drop(columns=[c for c in drop_cols if c in d.columns])
    
    # (2) 결측 처리
    d['children'] = d['children'].fillna(0)
    d['country']  = d['country'].fillna('Unknown')
    d['agent']    = d['agent'].fillna(0)   # 0 = 여행사 미경유
    
    return d

train_p = preprocess(train)
valid_p = preprocess(valid)
test_p  = preprocess(test)

print("컬럼 수:", train_p.shape[1])
print("결측:", train_p.isna().sum().sum())
print(train_p.columns.tolist())

컬럼 수: 23


결측:

 0
['hotel', 'is_canceled', 'lead_time', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'deposit_type', 'agent', 'customer_type', 'adr']


In [7]:
# ===== 4) agent 타깃 인코딩 (train 취소율 기반, 스무딩) =====
# 누수 방지: train에서만 취소율 계산 → valid/test에 매핑

def target_encode_agent(train, valid, test, col='agent', target='is_canceled', m=100):
    global_mean = train[target].mean()
    agg = train.groupby(col)[target].agg(['mean', 'count'])
    # 스무딩: 표본 적은 agent는 전체 평균 쪽으로
    smoothed = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    
    out = []
    for d in [train, valid, test]:
        d = d.copy()
        d['agent_te'] = d[col].map(smoothed).fillna(global_mean)  # 새 agent는 평균
        d = d.drop(columns=[col])
        out.append(d)
    return out

train_p, valid_p, test_p = target_encode_agent(train_p, valid_p, test_p)

print("agent_te 생성 확인:", 'agent_te' in train_p.columns)
print("agent 제거 확인:", 'agent' not in train_p.columns)
print("\nagent_te 분포 (train):")
print(train_p['agent_te'].describe().round(3))

agent_te 생성 확인: True
agent 제거 확인: True

agent_te 분포 (train):
count    83580.000
mean         0.383
std          0.165
min          0.073
25%          0.223
50%          0.395
75%          0.413
max          0.821
Name: agent_te, dtype: float64


In [8]:
# ===== 5) 범주형 처리 =====
cat_cols = ['hotel', 'meal', 'country', 'market_segment', 
            'distribution_channel', 'reserved_room_type', 
            'deposit_type', 'customer_type', 'arrival_date_month']

# country는 카디널리티 높음(178개) → 상위 N개만 남기고 나머지 'Other'
for d in [train_p, valid_p, test_p]:
    top_countries = train_p['country'].value_counts().head(20).index
    d['country'] = d['country'].where(d['country'].isin(top_countries), 'Other')

# category 타입으로 변환 (LightGBM이 직접 처리)
for d in [train_p, valid_p, test_p]:
    for c in cat_cols:
        d[c] = d[c].astype('category')

print("범주형 컬럼:", cat_cols)
print("\ncountry 고유값 수 (축소 후):", train_p['country'].nunique())
print("\n각 범주형 카디널리티:")
for c in cat_cols:
    print(f"  {c}: {train_p[c].nunique()}")

범주형 컬럼: ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type', 'arrival_date_month']

country 고유값 수 (축소 후): 21

각 범주형 카디널리티:
  hotel: 2
  meal: 5
  country: 21
  market_segment: 8
  distribution_channel: 5
  reserved_room_type: 10
  deposit_type: 3
  customer_type: 4
  arrival_date_month: 12


In [9]:
# ===== 6) X/y 분리 =====
target = 'is_canceled'

X_train, y_train = train_p.drop(columns=[target]), train_p[target]
X_valid, y_valid = valid_p.drop(columns=[target]), valid_p[target]
X_test,  y_test  = test_p.drop(columns=[target]),  test_p[target]

print("X_train:", X_train.shape)
print("피처:", X_train.columns.tolist())

X_train: (83580, 22)
피처: ['hotel', 'lead_time', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'deposit_type', 'customer_type', 'adr', 'agent_te']


In [10]:
# ===== 7) 다수 클래스 베이스라인 — 전부 "취소 안 함(0)"으로 찍기 =====
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np

majority_pred = np.zeros(len(y_test))
print("=== 다수 클래스 (전부 0) ===")
print(f"정확도: {accuracy_score(y_test, majority_pred):.3f}")
print(f"AUC: 0.500 (찍기라 순위 정보 없음)")
print(f"(참고: test 취소율 {y_test.mean():.3f}이라 전부 0으로 찍으면 정확도 {1-y_test.mean():.3f})")

=== 다수 클래스 (전부 0) ===
정확도: 0.704
AUC: 0.500 (찍기라 순위 정보 없음)
(참고: test 취소율 0.296이라 전부 0으로 찍으면 정확도 0.704)


In [11]:
# ===== 8) 로지스틱 회귀 베이스라인 =====
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# 범주형/수치형 컬럼 구분
cat_cols = ['hotel','meal','country','market_segment','distribution_channel',
            'reserved_room_type','deposit_type','customer_type','arrival_date_month']
num_cols = [c for c in X_train.columns if c not in cat_cols]

# 전처리: 수치형 스케일링 + 범주형 원핫
preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

logit = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

logit.fit(X_train, y_train)

# 평가
from sklearn.metrics import roc_auc_score
val_auc = roc_auc_score(y_valid, logit.predict_proba(X_valid)[:,1])
test_auc = roc_auc_score(y_test, logit.predict_proba(X_test)[:,1])
print(f"로지스틱  valid AUC: {val_auc:.4f}  |  test AUC: {test_auc:.4f}")

로지스틱  valid AUC: 0.8599  |  test AUC: 0.7578


In [12]:
# ===== 9) LightGBM 기본 =====
import lightgbm as lgb

# category 타입 그대로 사용 (LightGBM이 직접 처리)
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# 평가
from sklearn.metrics import roc_auc_score
val_auc = roc_auc_score(y_valid, model.predict(X_valid))
test_auc = roc_auc_score(y_test, model.predict(X_test))
print(f"\nLightGBM  valid AUC: {val_auc:.4f}  |  test AUC: {test_auc:.4f}")
print(f"(로지스틱: valid 0.8771 | test 0.7944)")

Training until validation scores don't improve for 50 rounds


[100]	train's auc: 0.935318	valid's auc: 0.895608


[200]	train's auc: 0.944347	valid's auc: 0.896781


[300]	train's auc: 0.950462	valid's auc: 0.897177


Early stopping, best iteration is:
[272]	train's auc: 0.94905	valid's auc: 0.897229



LightGBM  valid AUC: 0.8972  |  test AUC: 0.8270
(로지스틱: valid 0.8771 | test 0.7944)


In [13]:
# ===== 10) Feature Importance =====
import pandas as pd

imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
    'split': model.feature_importance(importance_type='split')
}).sort_values('gain', ascending=False)

print(imp.to_string(index=False))

                       feature          gain  split
                  deposit_type 162962.816075    121
                       country  77458.212907    612
                     lead_time  72317.497925   1436
                      agent_te  68515.146877    841
        previous_cancellations  32026.044071    223
                 customer_type  24645.421922    367
                market_segment  22333.249383    234
                           adr  19172.247176   1214
            arrival_date_month  10410.398808    482
      arrival_date_week_number   8417.964516    582
          stays_in_week_nights   6581.226930    383
previous_bookings_not_canceled   6531.895709    161
     arrival_date_day_of_month   6008.086393    543
       stays_in_weekend_nights   4241.940708    201
                         hotel   3787.605593    144
                          meal   3167.732233    155
                        adults   2840.026253    195
          distribution_channel   2104.804369     60
            

In [14]:
# ===== agent1 제외 실험 (공정 비교) =====
mask_train = (train['agent'] != 1.0).values
mask_valid = (valid['agent'] != 1.0).values
mask_test  = (test['agent'] != 1.0).values

# agent1 제외한 데이터로 학습
X_tr3, y_tr3 = X_train[mask_train], y_train[mask_train]
X_va3, y_va3 = X_valid[mask_valid], y_valid[mask_valid]
X_te3, y_te3 = X_test[mask_test], y_test[mask_test]

lgb_tr3 = lgb.Dataset(X_tr3, label=y_tr3)
lgb_va3 = lgb.Dataset(X_va3, label=y_va3, reference=lgb_tr3)
model3 = lgb.train(params, lgb_tr3, num_boost_round=1000,
    valid_sets=[lgb_va3], valid_names=['valid'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

# --- 비교 1: 각자의 test로 (문제 난이도 다름) ---
auc_full_on_full = roc_auc_score(y_test, model.predict(X_test))          # 전체모델 → 전체test
auc_no1_on_no1   = roc_auc_score(y_te3, model3.predict(X_te3))           # 제외모델 → 제외test

# --- 비교 2: 같은 test(agent1 제외)로 공정 비교 ---
auc_full_on_no1  = roc_auc_score(y_te3, model.predict(X_te3))            # 전체모델 → 제외test
auc_no1_on_no1_2 = roc_auc_score(y_te3, model3.predict(X_te3))           # 제외모델 → 제외test

print("=== 비교 1: 각자 test (난이도 다름) ===")
print(f"전체모델/전체test:   {auc_full_on_full:.4f}")
print(f"제외모델/제외test:   {auc_no1_on_no1:.4f}")
print("\n=== 비교 2: 같은 test(agent1제외)로 공정 비교 ===")
print(f"전체모델:  {auc_full_on_no1:.4f}")
print(f"제외모델:  {auc_no1_on_no1_2:.4f}")

Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[147]	valid's auc: 0.894813


=== 비교 1: 각자 test (난이도 다름) ===
전체모델/전체test:   0.8270
제외모델/제외test:   0.8267

=== 비교 2: 같은 test(agent1제외)로 공정 비교 ===
전체모델:  0.8269
제외모델:  0.8267


## 부록. 변수 제거가 성능에 미치는 영향 (LightGBM Test AUC)

위 파이프라인은 **예약 이후에 값이 쌓이는 변수 3개**(`booking_changes`, `total_of_special_requests`, `days_in_waiting_list`)와 `arrival_date_year`를 제외했다.
처음 버전(제외 전)과 비교해, 어떤 변수가 점수를 얼마나 올려 놓았는지 확인한다. 점수가 내려가는 만큼이 "예약 시점에 쓸 수 없는 정보가 만든 성능"이다.

In [15]:
POST_BOOKING = ['booking_changes', 'total_of_special_requests', 'days_in_waiting_list']
BASE_DROP = ['reservation_status', 'reservation_status_date', 'assigned_room_type', 'required_car_parking_spaces',
             'arrival_date', 'booking_date', 'booking_ym', 'company']

def run_variant(extra_drop):
    parts = []
    for d in (train, valid, test):
        d = d.drop(columns=[c for c in BASE_DROP + extra_drop if c in d.columns]).copy()
        d['children'] = d['children'].fillna(0); d['country'] = d['country'].fillna('Unknown'); d['agent'] = d['agent'].fillna(0)
        parts.append(d)
    tr, va, te = parts
    gm = tr['is_canceled'].mean(); agg = tr.groupby('agent')['is_canceled'].agg(['mean', 'count'])
    sm = (agg['mean'] * agg['count'] + gm * 100) / (agg['count'] + 100)
    for d in parts:
        d['agent_te'] = d['agent'].map(sm).fillna(gm); d.drop(columns=['agent'], inplace=True)
    top = tr['country'].value_counts().head(20).index
    cats = [c for c in cat_cols if c in tr.columns]
    for d in parts:
        d['country'] = d['country'].where(d['country'].isin(top), 'Other')
    for c in cats:
        cc = pd.Categorical(tr[c]).categories
        for d in parts: d[c] = pd.Categorical(d[c], categories=cc)
    X = [d.drop(columns=['is_canceled']) for d in parts]; y = [d['is_canceled'] for d in parts]
    dtr = lgb.Dataset(X[0], label=y[0]); dva = lgb.Dataset(X[1], label=y[1], reference=dtr)
    m = lgb.train(params, dtr, num_boost_round=1000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    return roc_auc_score(y[1], m.predict(X[1])), roc_auc_score(y[2], m.predict(X[2])), m.best_iteration, X[0].shape[1]

rows = []
for name, extra in [('제외 전 (원래 Part 2)', []), ('예약 이후 변수 3개 제외', POST_BOOKING),
                    ('arrival_date_year만 제외', ['arrival_date_year']), ('3개 + year 제외 (현재 Part 2)', POST_BOOKING + ['arrival_date_year'])]:
    v, t, it, nf_ = run_variant(extra)
    rows.append({'설정': name, '피처 수': nf_, 'Valid AUC': round(v, 4), 'Test AUC': round(t, 4), 'best_iter': it})
print(pd.DataFrame(rows).to_string(index=False))

                      설정  피처 수  Valid AUC  Test AUC  best_iter
        제외 전 (원래 Part 2)    26     0.9184    0.8560        291
          예약 이후 변수 3개 제외    23     0.9017    0.8319        275
   arrival_date_year만 제외    25     0.9156    0.8510        187
3개 + year 제외 (현재 Part 2)    22     0.8973    0.8271        272
